# Notebook 21: a stock chemprop network trained the way the rank-19 and rank-25 entries train

Lever 3 in PLAN.md. Reading `Scigantic/open-admet-challenge-cyp` (MIT-0, rank 25 at 0.4436)
showed that their reproduction of the rank-19 Workbench recipe differs from our `mtl13` in
how it is *trained*, not only in its data:

| | `mtl13` (nb5) | this notebook |
| :--- | :--- | :--- |
| loss | squared error | **MAE**, or the **interval hinge** on the scored heads |
| length | fixed 25 epochs, no validation | **early stopping** on a scaffold-held validation slice |
| schedule | AdamW, constant 3e-4 | chemprop's warmup then 1e-4 to 1e-3 to 1e-4 |
| test predictions | one model refit on all dev rows | **average of the five fold models** |
| head weights | equal | scored 1.0, auxiliary **0.3** |
| auxiliary heads | screen log2FC, AID 1851 AC50 (actives only) | screen log2FC, **ChEMBL IC50 x5**, AID 1851 **Max_Response x5** |
| encoder | `chemprop_medium` warm start | random init, d_h 700, depth 5, norm aggregation, batch norm |

The 18 heads match their `union18` member. The architecture and schedule follow the
Workbench recipe as Scigantic pinned it (`FROZEN_REPRODUCIBILITY_CONTRACT.json`), with two
changes for Colab time: batch 128 instead of 64, and at most 150 epochs with patience 25
instead of 400 and 40. **Credit: SuperCowPowers Workbench and Scigantic, in the method
report.**

Set `VARIANT` in the setup cell:

| `VARIANT` | scored-head loss | arm name |
| :--- | :--- | :--- |
| `'mae'` (default) | MAE | `cpmae` |
| `'iv'` | interval hinge `max(0, lo-p) + max(0, p-hi)` in pIC50 units | `cpiv` |

Their measurement: hinge against MAE at uniform weights, +0.0275 macro OOF ST-RAE, helping
1A2, 2C9 and 3A4 and not 2D6. Sample weights here are uniform, since they found
inverse-variance weights hurt the hinge.

**Folds.** The five dev folds are nb5's `GroupKFold(5)` on scaffold groups, so the
out-of-fold Spearman is comparable with `mtl13` (0.5577 / 0.6935 / 0.4515 / 0.8022, macro
dev 0.6262, macro holdout 0.6652). Inside each fold, about 10% of the remaining dev rows,
held out by scaffold, become the early-stopping validation slice, so each model trains on
roughly 72% of dev rows against `mtl13`'s 80%. Holdout and test predictions are the mean
of the five fold models; no model ever sees a holdout row.

**Gate**, same as nb20: macro dev Spearman at least +0.015 over `mtl13`, then a blend test
against `v25`.

**Safe to run beside notebooks 19 and 20**, and the two variants beside each other, each
in its own Colab runtime. It reads none of their files. Everything it writes carries its
own name: `nb21_<VARIANT>_f<k>.npz` per fold, `<arm>_<iso>.npz`,
`results_nb21_<VARIANT>.json`, and its own caches `aid1851_nb21.csv` and
`chembl_nb21.csv`, each written to a temporary name and renamed. Lightning checkpoints go
to the runtime's local disk only. The results zip holds only this run's files.

**Runtime: not measured on a GPU.** Each fold prints its epoch count and minutes; budget
roughly 30 to 60 minutes per fold on a T4. Run all resumes at the next unfinished fold
after a disconnect.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
VARIANT = 'mae'         # 'mae' or 'iv'; see the table above
ARM = {'mae': 'cpmae', 'iv': 'cpiv'}[VARIANT]
MAX_EPOCHS, PATIENCE, BATCH = 150, 25, 128
AUX_WEIGHT = 0.3
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')
OUTDIR = '/content/drive/MyDrive/cyp_results'
os.makedirs(OUTDIR, exist_ok=True)
print('variant', VARIANT, '| arm', ARM, '| writing to', OUTDIR)

NEED = ['meta_train.csv.gz', 'meta_test.csv.gz', 'holdout_mask.npy', 'scaffold_groups.npy']
have = sorted(os.listdir('/content/cache'))
missing = [f for f in NEED if f not in have]
assert not missing, 'payload is missing %s. Found: %s' % (missing, have)

In [ ]:
!pip -q install chemprop==2.3.1

In [ ]:
import gc, json, os, shutil, time, urllib.request, uuid, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

CACHE=Path('/content/cache'); OUT=Path(OUTDIR)
ISO=['CYP1A2','CYP2C9','CYP2D6','CYP3A4']; PAN=['CYP1A2','CYP2C9','CYP2C19','CYP2D6','CYP3A4']
GPU=torch.cuda.is_available()
meta=pd.read_csv(CACHE/'meta_train.csv.gz'); mete=pd.read_csv(CACHE/'meta_test.csv.gz')
hold=np.load(CACHE/'holdout_mask.npy'); G=np.load(CACHE/'scaffold_groups.npy',allow_pickle=True)
SM_TR=meta['SMILES'].to_numpy(); SM_TE=mete['SMILES'].to_numpy()
assert len(meta)==len(hold)==len(G)

def cached(name, url, need_cols):
    """Download once to Drive under this notebook's own name. Written to a temporary
    file and renamed, so a half-written copy is never read, by this run or another."""
    p = OUT / name
    def ok(q):
        try:
            return set(need_cols) <= set(pd.read_csv(q, nrows=0).columns)
        except Exception:
            return False
    if not (p.exists() and ok(p)):
        print('downloading', name)
        tmp = Path('/content') / (name + '.dl')
        urllib.request.urlretrieve(url, str(tmp))
        assert ok(tmp), f'{name}: download lacks {need_cols}'
        part = OUT / f'{name}.{uuid.uuid4().hex}.tmp'
        shutil.copyfile(tmp, part); os.replace(part, p)
    return p

AID = cached('aid1851_nb21.csv',
    'https://pubchem.ncbi.nlm.nih.gov/assay/pcget.cgi?query=download'
    '&record_type=datatable&actvty=all&response_type=save&aid=1851',
    ['Max_Response', 'Panel Name', 'PUBCHEM_EXT_DATASOURCE_SMILES'])
CHB = cached('chembl_nb21.csv',
    'https://raw.githubusercontent.com/jeremycheminf/openadmet_scripts/main/'
    'CYP_Challenge/data/external/cyp_chembl.csv',
    ['smiles'] + [f'{p}_IC50_PIC50' for p in PAN])

raw=pd.read_csv(AID, low_memory=False, skiprows=[1])
PANEL={'p450-cyp1a2':'CYP1A2','p450-cyp2c9':'CYP2C9','p450-cyp2c19':'CYP2C19',
       'p450-cyp2d6':'CYP2D6','p450-cyp3a4':'CYP3A4'}
raw['panel']=raw['Panel Name'].map(PANEL)
# Negated so larger means more inhibition; clipped because a few rows reach +1330.
raw['mr']=(-pd.to_numeric(raw['Max_Response'], errors='coerce')).clip(-50, 150)
raw=raw.dropna(subset=['panel','mr','PUBCHEM_EXT_DATASOURCE_SMILES'])
chembl=pd.read_csv(CHB)
print('AID 1851 rows', len(raw), '| ChEMBL rows', len(chembl))

## One row per compound, 18 heads

Public structures are reduced to their largest fragment and keyed by the first block of the
InChIKey, so salts, stereo and charge variants of one compound collapse to one row. Any key
that matches a challenge training or blind compound is dropped, which is stricter than the
canonical-SMILES check nb5 used; the rank-25 entry found one blind compound that a
SMILES or name check had missed.

In [ ]:
def key_and_smiles(s):
    m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
    if m is None:
        return None, None
    frags = Chem.GetMolFrags(m, asMols=True)
    m = max(frags, key=lambda f: f.GetNumHeavyAtoms())
    k = Chem.MolToInchiKey(m)
    return (k.split('-')[0] if k else None), Chem.MolToSmiles(m)

ours = set()
for s in np.concatenate([SM_TR, SM_TE]):
    k, _ = key_and_smiles(s)
    if k: ours.add(k)
for col in ('conn_key',):
    for frame in (meta, mete):
        if col in frame:
            ours |= {str(v).split('-')[0] for v in frame[col].dropna()}
print('challenge connectivity keys', len(ours))

def keyed(smiles):
    ks = {s: key_and_smiles(s) for s in pd.unique(smiles)}
    return [ks[s][0] for s in smiles], [ks[s][1] for s in smiles]

raw['key'], raw['smi'] = keyed(raw['PUBCHEM_EXT_DATASOURCE_SMILES'].to_numpy())
chembl['key'], chembl['smi'] = keyed(chembl['smiles'].to_numpy())
raw = raw.dropna(subset=['key']); chembl = chembl.dropna(subset=['key'])
mr = raw.pivot_table(index='key', columns='panel', values='mr', aggfunc='median').reindex(columns=PAN)
ch = chembl.groupby('key')[[f'{p}_IC50_PIC50' for p in PAN]].median()
ch.columns = PAN
pub_keys = sorted((set(mr.index) | set(ch.index)) - ours)
n_drop = len(set(mr.index) | set(ch.index)) - len(pub_keys)
smi_of = dict(zip(raw['key'], raw['smi'])); smi_of.update(zip(chembl['key'], chembl['smi']))
print(f'public compounds {len(pub_keys)} (dropped {n_drop} that match a challenge compound)')

HEADS = ([f'chal_{i}' for i in ISO] + [f'scr_{i}' for i in ISO]
         + [f'ch_{p}' for p in PAN] + [f'mr_{p}' for p in PAN])
NT = len(HEADS)
n_ch = len(SM_TR); n_pub = len(pub_keys)
SM_ALL = np.concatenate([SM_TR, np.array([smi_of[k] for k in pub_keys], dtype=object)])
Y = np.full((n_ch + n_pub, NT), np.nan)
for j, i in enumerate(ISO):
    Y[:n_ch, j] = meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float)
    Y[:n_ch, 4 + j] = meta[f'{i}_log2fc'].to_numpy(float)
Y[n_ch:, 8:13] = ch.reindex(pub_keys).to_numpy(float)
Y[n_ch:, 13:18] = mr.reindex(pub_keys).to_numpy(float)

# Credible-interval bounds for the scored heads ride as extra target columns (task weight
# 0) so the interval loss can see them. Where a label has no bound, the bound collapses to
# the label and the hinge reduces to MAE.
LO = np.column_stack([meta[f'{i}_pIC50_direct_inhibition_conf_low'].to_numpy(float) for i in ISO])
HI = np.column_stack([meta[f'{i}_pIC50_direct_inhibition_conf_high'].to_numpy(float) for i in ISO])
lab = np.isfinite(Y[:n_ch, :4])
nob = lab & ~(np.isfinite(LO) & np.isfinite(HI))
LO = np.where(nob, Y[:n_ch, :4], LO); HI = np.where(nob, Y[:n_ch, :4], HI)
LO[~lab] = np.nan; HI[~lab] = np.nan
B = np.full((n_ch + n_pub, 8), np.nan); B[:n_ch, :4] = LO; B[:n_ch, 4:] = HI
YA = np.hstack([Y, B]) if VARIANT == 'iv' else Y
NAUG = YA.shape[1]
TW = np.r_[np.ones(4), np.full(NT - 4, AUX_WEIGHT), np.zeros(NAUG - NT)]
print('table', YA.shape, '| labels per head:',
      dict(zip(HEADS, np.isfinite(Y).sum(0).tolist())))
print('labels without a credible interval (bound set to the label):', int(nob.sum()))

In [ ]:
import logging, warnings
import lightning as L
for _n in ('lightning', 'lightning.pytorch', 'lightning.fabric'):
    logging.getLogger(_n).setLevel(logging.ERROR)
warnings.filterwarnings('ignore', module='lightning')
from lightning.pytorch.callbacks import Callback, EarlyStopping, ModelCheckpoint
from chemprop import data, featurizers, models, nn
from chemprop.nn.metrics import ChempropMetric

class IntervalMAE(ChempropMetric):
    """MAE on every head, except the scored heads, which take the credible-interval hinge
    in real pIC50 units. Port of the rank-25 entry's train_interval.py."""
    def __init__(self, mean, scale, task_weights):
        super().__init__(task_weights=task_weights)
        self.register_buffer('mu', torch.as_tensor(mean, dtype=torch.float))
        self.register_buffer('sd', torch.as_tensor(scale, dtype=torch.float))

    def update(self, preds, targets, mask=None, weights=None, lt_mask=None, gt_mask=None):
        if mask is not None:
            mask = mask.clone(); mask[:, NT:] = False   # bounds are not observations
        super().update(preds, targets, mask, weights, lt_mask, gt_mask)

    def _calc_unreduced_loss(self, preds, targets, *args):
        cols = [(preds[:, j] - targets[:, j]).abs() for j in range(preds.shape[1])]
        for j in range(4):
            lo_j, hi_j = NT + j, NT + 4 + j
            p = preds[:, j] * self.sd[j] + self.mu[j]
            lo = targets[:, lo_j] * self.sd[lo_j] + self.mu[lo_j]
            hi = targets[:, hi_j] * self.sd[hi_j] + self.mu[hi_j]
            cols[j] = (torch.relu(lo - p) + torch.relu(p - hi)) / self.sd[j]
        return torch.stack(cols, dim=1)

def make(scaler):
    mp = nn.BondMessagePassing(d_h=700, depth=5, dropout=0.1)
    tw = torch.as_tensor(TW, dtype=torch.float)
    crit = (IntervalMAE(scaler.mean_, scaler.scale_, tw) if VARIANT == 'iv'
            else nn.metrics.MAE(task_weights=tw))
    ffn = nn.RegressionFFN(n_tasks=NAUG, input_dim=700, hidden_dim=300, n_layers=2,
                           dropout=0.1, criterion=crit,
                           output_transform=nn.UnscaleTransform.from_standard_scaler(scaler))
    return models.MPNN(mp, nn.NormAggregation(norm=100), ffn, batch_norm=True,
                       metrics=[nn.metrics.MAE()], warmup_epochs=2,
                       init_lr=1e-4, max_lr=1e-3, final_lr=1e-4)

class Every(Callback):
    def __init__(self, n=10): self.n, self.t0 = n, time.time()
    def on_validation_epoch_end(self, trainer, pl_module):
        e = trainer.current_epoch
        v = trainer.callback_metrics.get('val_loss')
        if not trainer.sanity_checking and v is not None and e % self.n == 0:
            print(f'    epoch {e:3d}  val_loss {float(v):.4f}  {(time.time()-self.t0)/60:.1f} min',
                  flush=True)

FZ = featurizers.SimpleMoleculeMolGraphFeaturizer()
PTS = [data.MoleculeDatapoint.from_smi(s, YA[i]) for i, s in enumerate(SM_ALL)]
PTS_TE = [data.MoleculeDatapoint.from_smi(s) for s in SM_TE]

def dl(idx, shuffle=False, scaler=None, pts=None):
    ds = data.MoleculeDataset(pts if pts is not None else [PTS[i] for i in idx], FZ)
    ds.cache = True    # featurise once per fold, not once per epoch
    sc = None
    if scaler == 'fit':
        sc = ds.normalize_targets()
    elif scaler is not None:
        ds.normalize_targets(scaler)
    return data.build_dataloader(ds, batch_size=BATCH, num_workers=0, shuffle=shuffle), sc

def predict(model, loader):
    pr = L.Trainer(accelerator='auto', devices=1, logger=False, enable_progress_bar=False,
                   enable_model_summary=False,
                   precision='16-mixed' if GPU else 32)
    return np.concatenate([p.float().numpy() for p in pr.predict(model, loader)])[:, :4]

## Five folds

Per fold: train on the other dev rows plus every public row, stop early on a scaffold-held
10% validation slice, keep the best epoch, and predict that fold's dev rows, the holdout and
the blind set. A fold that stops inside its first 10 epochs is treated as diverged (the
rank-25 entry lost a fold to a NaN validation loss that still produced plausible-looking
predictions) and the run halts.

In [ ]:
y_all = {i: meta[f'{i}_pIC50_direct_inhibition'].to_numpy(float) for i in ISO}
obs_any = np.isfinite(np.column_stack([y_all[i] for i in ISO])).any(1)
dev_rows = np.flatnonzero(obs_any & ~hold); ho_rows = np.flatnonzero(obs_any & hold)
pub_idx = np.arange(n_ch, n_ch + n_pub)
LOG = []

def fold_file(k): return OUT / f'nb21_{VARIANT}_f{k}.npz'

t_all = time.time()
for k, (tr, va) in enumerate(GroupKFold(5).split(dev_rows, dev_rows, G[dev_rows])):
    if fold_file(k).exists():
        print(f'fold {k}: already done, skipping'); continue
    t0 = time.time()
    inner = dev_rows[tr]
    a, b = next(GroupShuffleSplit(1, test_size=0.10, random_state=42 + k)
                .split(inner, inner, G[inner]))
    trn = np.concatenate([inner[a], pub_idx]); val = inner[b]
    L.seed_everything(42 + k, workers=True)
    tl, sc = dl(trn, shuffle=True, scaler='fit')
    vl, _ = dl(val, scaler=sc)
    model = make(sc)
    ckdir = Path(f'/content/nb21_ckpt/{VARIANT}_f{k}'); shutil.rmtree(ckdir, ignore_errors=True)
    ck = ModelCheckpoint(dirpath=str(ckdir), monitor='val_loss', mode='min', save_top_k=1)
    trainer = L.Trainer(accelerator='auto', devices=1, max_epochs=MAX_EPOCHS, logger=False,
                        enable_progress_bar=False, enable_model_summary=False,
                        precision='16-mixed' if GPU else 32,
                        callbacks=[ck, EarlyStopping(monitor='val_loss', patience=PATIENCE,
                                                     mode='min'), Every(10)])
    trainer.fit(model, tl, vl)
    best_epoch = int(Path(ck.best_model_path).stem.split('epoch=')[1].split('-')[0])
    assert trainer.current_epoch >= min(10, MAX_EPOCHS - 1), (
        f'fold {k} stopped at epoch {trainer.current_epoch}: diverged (check for NaN '
        'val_loss). Delete nothing; rerun this cell to retry the fold.')
    # A custom criterion cannot be rebuilt by load_from_checkpoint, so rebuild and load.
    best = make(sc)
    best.load_state_dict(torch.load(ck.best_model_path, map_location='cpu',
                                    weights_only=False)['state_dict'])
    best.eval()
    oof = predict(best, dl(dev_rows[va])[0])
    hp = predict(best, dl(ho_rows)[0])
    tp = predict(best, dl(None, pts=PTS_TE)[0])
    np.savez_compressed(fold_file(k), te=dev_rows[va], oof=oof, hp=hp, tp=tp,
                        epochs=trainer.current_epoch, best_epoch=best_epoch)
    r = [spearmanr(y_all[i][dev_rows[va]], oof[:, j], nan_policy='omit').statistic
         for j, i in enumerate(ISO)]
    print(f'fold {k}: {trainer.current_epoch} epochs (best {best_epoch}), '
          f'{(time.time()-t0)/60:.1f} min | dev rho ' + ' '.join(f'{x:.3f}' for x in r),
          flush=True)
    del model, best, trainer; gc.collect(); torch.cuda.empty_cache()
print(f'all folds done ({(time.time()-t_all)/60:.1f} min this session)')

In [ ]:
oof_all = np.full((n_ch, 4), np.nan); HP = []; TP = []; info = []
for k in range(5):
    z = np.load(fold_file(k))
    oof_all[z['te']] = z['oof']; HP.append(z['hp']); TP.append(z['tp'])
    info.append((int(z['epochs']), int(z['best_epoch'])))
hp = np.mean(HP, 0); tp = np.mean(TP, 0)
print('epochs per fold (run, best):', info)

RESULTS = []
for j, i in enumerate(ISO):
    y = y_all[i]
    d = dev_rows[np.isfinite(y[dev_rows])]; h = ho_rows[np.isfinite(y[ho_rows])]
    hsel = np.isin(ho_rows, h)
    np.savez_compressed(OUT / f'{ARM}_{i}.npz', oof=oof_all[d, j], dev=d,
                        hold=hp[hsel, j], ho=h, test=tp[:, j])
    r = float(spearmanr(y[d], oof_all[d, j]).statistic)
    rh = float(spearmanr(y[h], hp[hsel, j]).statistic)
    RESULTS.append(dict(iso=i, dev_spearman=r, holdout_spearman=rh))
    print(f'{i}: dev rho {r:.4f} | holdout rho {rh:.4f}', flush=True)
json.dump(dict(variant=VARIANT, arm=ARM, epochs=info, results=RESULTS),
          open(OUT / f'results_nb21_{VARIANT}.json', 'w'), indent=2)
print('\nmtl13 (nb5): 0.5577 0.6935 0.4515 0.8022 | macro dev 0.6262 | macro holdout 0.6652')
print('%s macro dev %.4f | macro holdout %.4f' % (ARM,
    np.mean([x['dev_spearman'] for x in RESULTS]),
    np.mean([x['holdout_spearman'] for x in RESULTS])))

# Only this run's own files; never the whole folder, which nb19 or nb20 may be writing to.
import zipfile
mine = ([fold_file(k) for k in range(5)] + [OUT / f'results_nb21_{VARIANT}.json']
        + [OUT / f'{ARM}_{i}.npz' for i in ISO])
with zipfile.ZipFile(f'/content/nb21_{VARIANT}_results.zip', 'w') as z:
    for p in mine:
        z.write(p, p.name)
try:
    from google.colab import files
    files.download(f'/content/nb21_{VARIANT}_results.zip')
except Exception as e:
    print('download skipped:', type(e).__name__, '- results are on Drive in', OUT)